In [12]:
# Cell 1 - Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [13]:
# Cell 2 - Rename uploaded files to the names expected by Cell B0
from pathlib import Path

RENAME_MAP = [
    ("directive_no",      "mkn_directive_20_1997.pdf"),
    ("national_security", "mkn_directive_20_1997.pdf"),
    ("1997-policy",       "mkn_directive_20_1997.pdf"),
    ("garis_panduan_pps", "nadma_garis_panduan_pps_my.pdf"),
    ("arahan_nadma",      "arahan_nadma_no1_2024_my.pdf"),
    ("pto_bencana",       "jkm_pto_bencana_2018_my.pdf"),
    ("ets_8",             "nadma_ets8_peranan_sukarelawan_my.pdf"),
]
final_names = {new for _, new in RENAME_MAP}

for f in sorted(RAW_DIR.glob("*.pdf")):
    if f.name in final_names:
        continue
    for keyword, new_name in RENAME_MAP:
        if keyword in f.name.lower():
            target = RAW_DIR / new_name
            if target.exists():
                print("Already exists, skipped:", f.name)
            else:
                f.rename(target)
                print("Renamed:", f.name, "->", new_name)
            break
    else:
        print("No match (leave it out):", f.name)

No match (leave it out): PTO Bencana 2018(1).pdf


In [14]:
# Cell 3 - Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 4
PTO Bencana 2018(1).pdf bytes: 4242018
arahan_nadma_no1_2024_my.pdf bytes: 2101197
mkn_directive_20_1997.pdf bytes: 192788
nadma_garis_panduan_pps_my.pdf bytes: 230221
Upload verified. Select your relevant input(s) for Step A3.


In [15]:
# Cell 4 - REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [16]:
# Cell B0 - PROJECT CONFIGURATION AND DOCUMENT HELPERS (ReliefOps QA)
# Run after Cell 4A. Run again after any runtime reset (then rerun Cells 1, 3, 4, 4A first).
!pip -q install pypdf

from pathlib import Path
import re
import json
import unicodedata
from pypdf import PdfReader

# ---------- 1. Approved sources (file names must match the uploaded files) ----------
ALL_SOURCES = {
    "S01": "mkn_directive_20_1997.pdf",
    "S02": "nadma_garis_panduan_pps_my.pdf",
    "S03": "arahan_nadma_no1_2024_my.pdf",
    "S04": "jkm_pto_bencana_2018_my.pdf",
    "S05": "nadma_ets8_peranan_sukarelawan_my.pdf",   # optional: volunteer slides
}
# Only sources that are really uploaded are processed (S05 is skipped if absent)
SOURCES = {k: v for k, v in ALL_SOURCES.items() if (RAW_DIR / v).is_file()}

# ---------- 2. Preparation configuration ----------
CHUNK_SIZE = 500
PROCESSING_LABEL = "page-aware-paragraph-char-500"

WHOLE_PAGES = {"S02": [4]}                                   # flowchart page kept as one chunk
TOC_PAGES = {                                                # contents pages (excluded)
    "S01": [1, 2],
    "S03": [1, 2, 3, 4, 6, 7, 8, 78, 79, 80],
    "S04": [1, 2, 4, 6, 14, 18, 36, 56, 70, 84, 87, 88],
}
SKIP_PAGES = {                                               # forms and annexes (excluded)
    "S04": [51, 52, 53, 64, 65, 66, 67, 76, 77, 78, 79, 80, 81],
}
KEEP_PAGES = {"S05": list(range(53, 67))}                    # only these pages are used
MIN_PAGE_CHARS = {"S03": 100, "S04": 100, "S05": 60}         # shorter pages are dropped
DEFAULT_MIN_CHARS = 30
PAGE_TRIM_BEFORE = {("S03", 9): r"HURUF RINGKAS"}            # drop text before this marker

SOURCE_FLAGS = {"S01": ["superseded"], "S02": ["covid_era"], "S05": ["slide_text"]}
COVID_TERMS = (r"covid|\bpkp\b|\bppe\b|saringan kesihatan|topeng muka|pelitup muka|"
               r"mysejahtera|thermometer|disinfeksi|hand sanitizer")

NOISE_RULES = {
    "S03": [r"(?im)^\s*TERHAD\s*$",
            r"(?im)^.*teks NADMA New\.indd\s+\d+.*$",
            r"(?m)^\s*(?:[ivxlc]+|\d{1,3})\s*$"],
    "S04": [r"(?im)^\s*TERHAD\s*$",
            r"(?m)^\s*Peraturan Tetap Operasi\s+Pengurusan Bencana, JKM\s*$",
            r"(?m)^\s*2018\s*\|\s*\d+\s*$",
            r"(?m)^\s*\d+\s*\|\s*2018\s*$"],
    "S05": [r"AGENSI PENGURUSAN BENCANA NEGARA",
            r"JABATAN\s*PERDANA MENTERI",
            r"(?m)^\s*\d{1,3}\s*$"],
}

# ---------- 3. Text helpers ----------
def extract_pages(path):
    reader = PdfReader(str(path))
    return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]

def normalise(text):
    text = unicodedata.normalize("NFKC", text).replace("\xa0", " ")
    # stray "E" produced by a closing quote in S01, e.g. "post-mortemEis" -> "post-mortem' is"
    text = re.sub(r"(‘[^‘’\n]{1,40}?)E(?=[A-Za-z])", r"\1’ ", text)
    # spaced hyphen, e.g. "badan -badan" -> "badan-badan", "COVID -19" -> "COVID-19"
    text = re.sub(r"(?<=\w) -(?=\w)", "-", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text

def clean_noise(sid, text):
    for pattern in NOISE_RULES.get(sid, []):
        text = re.sub(pattern, "", text)
    return text

PARA_START = re.compile(
    r"^(?:\d{1,3}\.\s|\(?[a-z]{1,2}\)\s|\(?[ivx]+\)\s|[ivx]+\.\s|[•–-]\s|"
    r"PERKARA\b|LAMPIRAN\b|BAHAGIAN\b)")
HEADING = re.compile(r"^[A-Z0-9][A-Z0-9 ,/&()\-:.’‘]{7,}$")

def reflow(text):
    """Rebuild paragraphs from hard-wrapped PDF lines. Paragraphs are separated by a blank line."""
    paras, current = [], ""
    for raw in text.split("\n"):
        line = raw.strip()
        if not line:
            continue
        if HEADING.match(line):
            if current:
                paras.append(current)
            paras.append(line)
            current = ""
        elif PARA_START.match(line) and current:
            paras.append(current)
            current = line
        elif not current:
            current = line
        elif current.endswith("-") and line[:1].islower():
            current += line
        else:
            current += " " + line
    if current:
        paras.append(current)
    return "\n\n".join(re.sub(r"\s+", " ", p).strip() for p in paras if p.strip())

def is_toc(text):
    head = text[:250].lower()
    if "kandungan" in head or head.strip().startswith("contents") or "table of contents" in head:
        return True
    lines = [l for l in text.split("\n") if l.strip()]
    if len(lines) < 6:
        return False
    dotted = sum(bool(re.search(r"(\.{4,}|\s\d{1,3}$)", l.strip())) for l in lines)
    return dotted / len(lines) > 0.6

def chunk_flags(sid, text):
    flags = list(SOURCE_FLAGS.get(sid, []))
    if re.search(COVID_TERMS, text, flags=re.I):
        flags.append("covid_health")
    return flags

# ---------- 4. Page selection and cleaning ----------
def clean_pages(sid, path):
    """Return [(page_no, cleaned_text, whole_page_flag)] after all skip rules, plus a stats dict."""
    kept, stats = [], {"pages_total": 0, "skipped_toc": 0, "skipped_other": 0, "skipped_short": 0}
    for page_no, raw in extract_pages(path):
        stats["pages_total"] += 1
        if sid in KEEP_PAGES and page_no not in KEEP_PAGES[sid]:
            stats["skipped_other"] += 1
            continue
        if page_no in SKIP_PAGES.get(sid, []):
            stats["skipped_other"] += 1
            continue
        if page_no in TOC_PAGES.get(sid, []) or is_toc(raw):
            stats["skipped_toc"] += 1
            continue
        text = normalise(raw)
        marker = PAGE_TRIM_BEFORE.get((sid, page_no))
        if marker:
            m = re.search(marker, text)
            if m:
                text = text[m.start():]
        text = clean_noise(sid, text)
        whole = page_no in WHOLE_PAGES.get(sid, [])
        text = " | ".join(l.strip() for l in text.split("\n") if l.strip()) if whole else reflow(text)
        if len(text) < MIN_PAGE_CHARS.get(sid, DEFAULT_MIN_CHARS):
            stats["skipped_short"] += 1
            continue
        kept.append((page_no, text, whole))
    stats["pages_kept"] = len(kept)
    stats["characters"] = sum(len(t) for _, t, _ in kept)
    return kept, stats

# ---------- 5. Chunking ----------
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

def split_long(segment, size):
    """Split an over-long paragraph at sentence boundaries, then at words."""
    out, current = [], ""
    for sentence in re.split(r"(?<=[.;:!?])\s+", segment):
        if len(sentence) > size:
            if current:
                out.append(current)
                current = ""
            words, buf = sentence.split(" "), ""
            for w in words:
                if buf and len(buf) + 1 + len(w) > size:
                    out.append(buf)
                    buf = w
                else:
                    buf = (buf + " " + w).strip()
            current = buf
        elif current and len(current) + 1 + len(sentence) > size:
            out.append(current)
            current = sentence
        else:
            current = (current + " " + sentence).strip()
    if current:
        out.append(current)
    return out

def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts, current = [], ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(split_long(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

def page_aware_chunks(pages, size=500):
    """Paragraph chunking inside each page. Returns [(page_no, chunk_text)]."""
    out = []
    for page_no, text, whole in pages:
        chunks = [text] if whole else paragraph_chunks(text, size)
        out.extend((page_no, c) for c in chunks if c.strip())
    return out

def build_records(sid, path, size=CHUNK_SIZE, label=PROCESSING_LABEL):
    """Single source of truth used by Part B (Cell 5A2) and Part C (Cell 8A)."""
    path = Path(path)
    pages, stats = clean_pages(sid, path)
    digest = source_hash(path)
    counters, records = {}, []
    for page_no, chunk in page_aware_chunks(pages, size):
        counters[page_no] = counters.get(page_no, 0) + 1
        records.append({
            "record_id": f"{sid}-p{page_no:03d}-{counters[page_no]:02d}",
            "source_id": sid,
            "source": path.name,
            "source_sha256": digest,
            "page": page_no,
            "chunk_index": counters[page_no],
            "text": chunk,
            "processing": label,
            "flags": chunk_flags(sid, chunk),
        })
    return records, stats

print("Cell B0 ready.")
print("Sources found:", {k: v for k, v in SOURCES.items()})
missing = [k for k in ALL_SOURCES if k not in SOURCES]
print("Not uploaded (skipped):", missing if missing else "none")

Cell B0 ready.
Sources found: {'S01': 'mkn_directive_20_1997.pdf', 'S02': 'nadma_garis_panduan_pps_my.pdf', 'S03': 'arahan_nadma_no1_2024_my.pdf'}
Not uploaded (skipped): ['S04', 'S05']


In [17]:
# Cell 5A1 - PATHWAY A1: inspect each source and compare three chunking methods
assert SOURCES, "No approved source found in data/raw. Check the file names in Cell B0."

comparison = {}
page_cache = {}

for sid, filename in SOURCES.items():
    path = select_source(relative_path=filename)
    pages, stats = clean_pages(sid, path)
    page_cache[sid] = (pages, stats)
    assert pages, f"{sid}: no usable text. Scanned PDFs may need OCR."

    full_text = "\n\n".join(text for _, text, _ in pages)
    fixed = fixed_chunks(full_text, CHUNK_SIZE)
    para = paragraph_chunks(full_text, CHUNK_SIZE)
    aware = page_aware_chunks(pages, CHUNK_SIZE)

    comparison[sid] = {
        "file": filename,
        "pages_total": stats["pages_total"],
        "pages_kept": stats["pages_kept"],
        "characters": stats["characters"],
        "fixed": len(fixed),
        "paragraph": len(para),
        "page_aware": len(aware),
        "avg_len_page_aware": round(sum(len(c) for _, c in aware) / len(aware)),
        "max_len_page_aware": max(len(c) for _, c in aware),
    }

print("--- Chunking comparison (chunk size = %d characters) ---" % CHUNK_SIZE)
print(f"{'ID':<4} {'pages':>9} {'chars':>8} {'fixed':>6} {'para':>6} {'page-aware':>10} {'avg':>5} {'max':>5}")
for sid, c in comparison.items():
    print(f"{sid:<4} {c['pages_kept']:>4}/{c['pages_total']:<4} {c['characters']:>8} "
          f"{c['fixed']:>6} {c['paragraph']:>6} {c['page_aware']:>10} "
          f"{c['avg_len_page_aware']:>5} {c['max_len_page_aware']:>5}")
tot = {k: sum(c[k] for c in comparison.values()) for k in ("fixed", "paragraph", "page_aware")}
print("TOTAL", tot)

# Representative samples (first source with enough text)
sid0 = next(iter(SOURCES))
pages0, _ = page_cache[sid0]
text0 = "\n\n".join(t for _, t, _ in pages0)
print(f"\nSample from {sid0}")
print("Fixed sample         :", repr(fixed_chunks(text0, CHUNK_SIZE)[3][:220]))
print("Paragraph sample     :", repr(paragraph_chunks(text0, CHUNK_SIZE)[3][:220]))
p, c = page_aware_chunks(pages0, CHUNK_SIZE)[3]
print(f"Page-aware sample (page {p}):", repr(c[:220]))

if "S02" in page_cache:
    print("\nS02 flowchart page kept as one chunk:")
    for p, t, whole in page_cache["S02"][0]:
        if whole:
            print(f"page {p}:", repr(t[:260]))

print("\nPATHWAY A1 completed successfully.")

--- Chunking comparison (chunk size = 500 characters) ---
ID       pages    chars  fixed   para page-aware   avg   max
S01    24/26      52722    106    134        139   378   500
S02     5/5        7423     15     19         19   389  1139
S03    70/80     247001    495    614        638   386   500
TOTAL {'fixed': 616, 'paragraph': 767, 'page_aware': 796}

Sample from S01
Fixed sample         : 'Kelang Port on June 20th. 1992 causing 10 death. The ship was carrying oil and such fire has the potentiality of spreading because it was close to Shell Company ,oil depot.\n\n4. The collapse of Highland Towers Condominium'
Paragraph sample     : '1. The collapse of Sultan Abdul Halim Ferry Terminal at Butterworth , Pulau Pinang on July 31st. 1998, which caused 32 death and 1674 others injured.\n\n2. An explosion and fire that ruined a fireworks factory belonging to'
Page-aware sample (page 3): '1. The collapse of Sultan Abdul Halim Ferry Terminal at Butterworth , Pulau Pinang on July 31st. 1

In [18]:
# Cell 5A2 - PATHWAY A2: export traceable page-aware records (Part B output)
import json

CHOSEN_METHOD = "page-aware"        # justified by the A1 comparison (page numbers are required for citations)
approach = PROCESSING_LABEL

all_records, per_source_counts = [], {}
for sid, filename in SOURCES.items():
    recs, stats = build_records(sid, RAW_DIR / filename, CHUNK_SIZE, approach)
    per_source_counts[sid] = len(recs)
    all_records.extend(recs)

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in all_records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert all(r["text"].strip() for r in all_records), "One or more chunks are empty"
assert len({r["record_id"] for r in all_records}) == len(all_records), "Duplicate record_id"
assert all(r["page"] and r["source_sha256"] for r in all_records), "Missing page or hash"

with prepared_path.open("r", encoding="utf-8") as file:
    exported = [json.loads(line) for line in file if line.strip()]
assert len(exported) == len(all_records), "Exported record count does not match"

flag_counts = {}
for r in all_records:
    for fl in r["flags"]:
        flag_counts[fl] = flag_counts.get(fl, 0) + 1

print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Records per source:", per_source_counts)
print("Total records:", len(all_records))
print("Flag counts:", flag_counts)
print("Prepared file:", prepared_path, "| size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
first = all_records[0]
print("First record:", {k: (v[:120] if k == "text" else v) for k, v in first.items()})

PATHWAY A2 completed successfully.
Selected approach: page-aware-paragraph-char-500
Records per source: {'S01': 139, 'S02': 19, 'S03': 638}
Total records: 796
Flag counts: {'superseded': 139, 'covid_era': 19, 'covid_health': 14}
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl | size: 533122 bytes
Validation: PASS
First record: {'record_id': 'S01-p003-01', 'source_id': 'S01', 'source': 'mkn_directive_20_1997.pdf', 'source_sha256': '1f9b2faf796291454d741a1b7d66abd3cebf7ed3132f2bef7ecda675ac336ee4', 'page': 3, 'chunk_index': 1, 'text': 'JBPM : Malaysian Fire and Rescue Department JPBBD : District Disaster Management and Relief Committee JPBBN : State Disa', 'processing': 'page-aware-paragraph-char-500', 'flags': ['superseded']}


In [19]:
# Step B3 - Basic output check
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

# Content check: one representative record from each source
import json
seen = {}
with prepared_path.open(encoding="utf-8") as fh:
    for line in fh:
        r = json.loads(line)
        seen.setdefault(r["source_id"], r)
for sid, r in seen.items():
    print(f"\n[{sid}] page {r['page']} | flags {r['flags']}")
    print(r["text"][:300])

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 533122
Validation: file exists and is non-empty; verify content manually too.

[S01] page 3 | flags ['superseded']
JBPM : Malaysian Fire and Rescue Department JPBBD : District Disaster Management and Relief Committee JPBBN : State Disaster Management and Relief Committee JPBBP : Central Disaster Management and Relief Committee JKR : Public Works Department JPM : Prime Minister Department LPTA : Atomic Energy Lic

[S02] page 1 | flags ['covid_era', 'covid_health']
Garis Panduan Pengurusan PPS semasa COVID-19 Selaras dengan arahan Perintah Kawalan Pergerakan Fasa-4 yang telah diumumkan, satu Garis Panduan khas bagi Pusat Pemindahan Sementara (PPS) dibangunkan untuk dijadikan rujukan berkenaan pengendalian PPS sepanjang operasi COVID-19. Garis Panduan ini melip

[S03] page 5 | flags []
PRAKATA Susulan daripada kejadian Pangsapuri Highland Towers di Hulu Klang, Selangor runtuh pada 11 Disember 1993, Jemaah Menteri pada

In [20]:
# B4.1 - Register Pathway A (one registration per source)
from pathlib import Path

# Re-initialize the common lineage registry if not already defined
if "lineage_entries" not in globals():
    lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")

    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print("Registered:", source.name, "->", output.name)

for sid, filename in SOURCES.items():
    register_lineage(
        source=RAW_DIR / filename,
        output=prepared_path,
        processing=f"{sid}: {approach}; noise removal; page-level provenance"
    )
print("Registered entries:", len(lineage_entries))

Registered: mkn_directive_20_1997.pdf -> prepared_text.jsonl
Registered: nadma_garis_panduan_pps_my.pdf -> prepared_text.jsonl
Registered: arahan_nadma_no1_2024_my.pdf -> prepared_text.jsonl
Registered entries: 3


In [21]:
# CELL 6A - GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 3
Saved to: /content/week3_project/data/week3_manifest.json


In [22]:
# CELL 6B - VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 3
Manifest: /content/week3_project/data/week3_manifest.json


In [23]:
# Cell 7 - REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified - check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [24]:
# Cell 8A - PDF/TXT pipeline (one pipeline run per source)
# Requires Cell 7 and Cell B0. Uses the same build_records() as Part B.
import json

def make_config(sid):
    return {
        "source_id": sid,
        "chunk_size": CHUNK_SIZE,
        "method": "page-aware",
        "processing": PROCESSING_LABEL,
        "toc_pages": TOC_PAGES.get(sid, []),
        "skip_pages": SKIP_PAGES.get(sid, []),
        "keep_pages": KEEP_PAGES.get(sid, []),
        "whole_pages": WHOLE_PAGES.get(sid, []),
    }

def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() != ".pdf":
        raise ValueError("Expected a PDF source")
    pages, stats = clean_pages(cfg["source_id"], src)
    if not pages:
        raise ValueError("No extractable text found. Scanned PDFs may require OCR.")
    return {"source": src.name, **stats}

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    if int(cfg["chunk_size"]) <= 0:
        raise ValueError("Chunk size must be positive")
    records, _ = build_records(cfg["source_id"], src, int(cfg["chunk_size"]), cfg["processing"])
    if not records:
        raise ValueError("No text chunks produced")
    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for rec in records:
            handle.write(json.dumps(rec, ensure_ascii=False) + "\n")
    mapping = [{"source": str(src), "output": str(target), "records": len(records)}]
    return [target], mapping, f"{cfg['method']} paragraph chunks of {cfg['chunk_size']} characters"

part_c_runs = {}
for sid, filename in SOURCES.items():
    part_c_runs[sid] = register_and_run(
        f"documents_{sid.lower()}",
        resolve_source(filename),
        inspect_documents,
        prepare_documents,
        make_config(sid),
    )
print("Cell 8A completed successfully.")

SUCCESS: documents_s01 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s01/pipeline_manifest.json
SUCCESS: documents_s02 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s02/pipeline_manifest.json
SUCCESS: documents_s03 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s03/pipeline_manifest.json
Cell 8A completed successfully.


In [25]:
# Cell 8A-check - Part C output must equal Part B output
import json

part_b = {}
with (PROCESSED_DIR / "prepared_text.jsonl").open(encoding="utf-8") as fh:
    for line in fh:
        r = json.loads(line)
        part_b.setdefault(r["source_id"], []).append(r)

folders = [f"documents_{s.lower()}" for s in SOURCES]
total = 0
for sid, folder in zip(SOURCES, folders):
    out = PIPE_DIR / folder / "prepared_text.jsonl"
    manifest = PIPE_DIR / folder / "pipeline_manifest.json"
    assert out.is_file() and manifest.is_file(), f"Missing output for {sid}"
    with out.open(encoding="utf-8") as fh:
        rows = [json.loads(l) for l in fh if l.strip()]
    assert rows == part_b[sid], f"{sid}: Part C differs from Part B"
    total += len(rows)
    print(f"{sid}: {len(rows):>4} records | Part C == Part B | {out}")
print("Total Part C records:", total)
print("Sample record:", {k: (v[:100] if k == "text" else v) for k, v in rows[len(rows)//2].items()})

S01:  139 records | Part C == Part B | /content/week3_project/data/pipeline_v1/documents_s01/prepared_text.jsonl
S02:   19 records | Part C == Part B | /content/week3_project/data/pipeline_v1/documents_s02/prepared_text.jsonl
S03:  638 records | Part C == Part B | /content/week3_project/data/pipeline_v1/documents_s03/prepared_text.jsonl
Total Part C records: 796
Sample record: {'record_id': 'S03-p038-07', 'source_id': 'S03', 'source': 'arahan_nadma_no1_2024_my.pdf', 'source_sha256': 'cc648944894c2ea079af5eb8fec571d3b349250b55461ecfef4dbf3c1a99c24f', 'page': 38, 'chunk_index': 7, 'text': '18. Menilai laporan mengenai pengendalian dan pengurusan Bencana yang berlaku meliputi aspek operasi', 'processing': 'page-aware-paragraph-char-500', 'flags': []}


In [26]:
# Cell 9 - REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A-8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} - source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents_s01 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s01/pipeline_manifest.json
PASS: documents_s01 - source fingerprints, prepared outputs and mappings match
SUCCESS: documents_s02 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s02/pipeline_manifest.json
PASS: documents_s02 - source fingerprints, prepared outputs and mappings match
SUCCESS: documents_s03 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s03/pipeline_manifest.json
PASS: documents_s03 - source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [27]:
# Package outputs for download (raw PDFs are excluded)
import zipfile
from pathlib import Path

zip_path = Path("/content/week3_outputs.zip")
data_dir = BASE_DIR / "data"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(data_dir.rglob("*")):
        if p.is_file() and "raw" not in p.relative_to(data_dir).parts and ".ipynb_checkpoints" not in p.parts:
            z.write(p, Path("data") / p.relative_to(data_dir))
print("Created:", zip_path, zip_path.stat().st_size, "bytes")

from google.colab import files
files.download(str(zip_path))

Created: /content/week3_outputs.zip 133885 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>